# 🦯 Mushir — AI Mobility Assistant: Kaggle GPU Training Pipeline
**See Beyond Sight. Move with Confidence.**

This notebook trains and fine-tunes **YOLOv8n / YOLO11n** on Kaggle's free **NVIDIA GPU (T4 x 2 or P100)** using the 21-class Indian Footpath & Mobility Hazard dataset (`mushir.yaml`), and exports an optimized **INT8 quantized `.tflite` model** ready for your Android phone.

---

### ⚡ Essential Kaggle Settings Before Running:
In the right-hand **Notebook options** sidebar:
1. **Accelerator**: Select **GPU T4 x 2** (or **GPU P100**)
2. **Internet**: Toggle **Internet ON** (Required for `git clone` and `pip install`)

---

In [ ]:
# Step 1: Check Kaggle GPU Accelerator
!nvidia-smi

In [ ]:
# Step 2: Install Ultralytics and dependencies in Kaggle
!pip install -q ultralytics roboflow opencv-python-headless

import ultralytics
ultralytics.checks()

In [ ]:
# Step 3: Clone Mushir Repository into Kaggle Working Directory
import os
import shutil
from pathlib import Path

repo_dir = Path('/kaggle/working/Mushir')
if repo_dir.exists():
    shutil.rmtree(repo_dir)

!git clone https://github.com/ddevi69105-droid/Mushir.git /kaggle/working/Mushir
%cd /kaggle/working/Mushir

In [ ]:
# Step 4: Validate Dataset Split & Classes for Indian Streets
!python scripts/prepare_dataset.py --validate datasets/mushir

In [ ]:
# Step 5: Configure YAML path for Kaggle working environment
import yaml

yaml_path = '/kaggle/working/Mushir/datasets/mushir.yaml'
with open(yaml_path, 'r') as f:
    data_cfg = yaml.safe_load(f)

# Set absolute path so Kaggle's loader finds images seamlessly
data_cfg['path'] = '/kaggle/working/Mushir/datasets/mushir'

with open(yaml_path, 'w') as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print("Updated datasets/mushir.yaml path to:", data_cfg['path'])

In [ ]:
# Step 6: Train YOLOv8n on Kaggle GPU
from ultralytics import YOLO
import torch

# Use GPU 0 (or [0, 1] for dual T4)
device = 0 if torch.cuda.is_available() else 'cpu'
print(f"Training on device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

model = YOLO('yolov8n.pt')

# Train with walking-height augmentations
results = model.train(
    data='/kaggle/working/Mushir/datasets/mushir.yaml',
    epochs=80,
    imgsz=640,
    batch=16,
    device=device,
    # Augmentations for walking camera tilt, glare, and clutter
    degrees=7.0,
    translate=0.1,
    scale=0.35,
    perspective=0.0004,
    fliplr=0.5,
    mosaic=0.8,
    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.4,
    project='/kaggle/working/runs/detect',
    name='mushir_kaggle',
    save=True,
    exist_ok=True
)

print("Training complete! Best weights saved at: /kaggle/working/runs/detect/mushir_kaggle/weights/best.pt")

In [ ]:
# Step 7: Check Safety-Critical Hazard Recall (Open Manhole, Pothole, Stairs, Auto-rickshaws)
!yolo detect val model=/kaggle/working/runs/detect/mushir_kaggle/weights/best.pt data=/kaggle/working/Mushir/datasets/mushir.yaml split=test

In [ ]:
# Step 8: Export to INT8 Quantized TFLite for Android Mobile
best_weights = '/kaggle/working/runs/detect/mushir_kaggle/weights/best.pt'
trained_model = YOLO(best_weights)

print("Exporting to INT8 Quantized TFLite...")
exported_result = trained_model.export(format='tflite', imgsz=640, int8=True, nms=False)
print("Export output path:", exported_result)

In [ ]:
# Step 9: Copy Model to /kaggle/working/ for 1-Click Download
import glob
from IPython.display import FileLink, display

tflite_candidates = glob.glob('/kaggle/working/runs/detect/mushir_kaggle/weights/*tflite*/*.tflite') + \
                    glob.glob('/kaggle/working/runs/detect/mushir_kaggle/weights/*.tflite')

dest_path = '/kaggle/working/mushir_yolov8n.tflite'
if tflite_candidates:
    src = tflite_candidates[0]
    shutil.copy2(src, dest_path)
    size_mb = os.path.getsize(dest_path) / (1024 * 1024)
    print(f"\nSUCCESS! Model ready: {dest_path} ({size_mb:.2f} MB)")
    print("\nClick the link below to download your model directly:")
    display(FileLink('mushir_yolov8n.tflite'))
    print("\nYou can also download 'mushir_yolov8n.tflite' from the Kaggle Output panel on the right sidebar!")
else:
    print("Could not locate exported .tflite. Please check /kaggle/working/runs/detect/mushir_kaggle/weights/")